[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/35_bpe.ipynb)

# 🔴 Hard: Byte-Pair Encoding (BPE)

Implement a simple **BPE tokenizer** — the foundation of GPT/LLaMA tokenization.

### Signature
```python
class SimpleBPE:
    def __init__(self): ...
    def train(self, corpus: list[str], num_merges: int): ...
    def encode(self, text: str) -> list[str]: ...
```

### Algorithm (training)
1. Split each word into characters + `</w>` end marker
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a single token
4. Repeat for `num_merges` iterations

In [30]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
# No imports needed

In [33]:
# ✏️ YOUR IMPLEMENTATION HERE

class SimpleBPE:
    def __init__(self):
        self.merges = []

    def train(self, corpus, num_merges):
      vocab={}
      for word in corpus:
        symbols=tuple(word) + ('',)
        vocab[symbols]=vocab.get(symbols,0)+1
      self.merges = []
      for _ in range(num_merges):
        pairs={}
        for word,freq in vocab.items():
          for i in range(len(word)-1):
            pair=(word[i],word[i+1])
            pairs[pair]=pairs.get(pair,0)+freq
        if not pairs:
                break
        best=max(pairs,key=pairs.get)
        self.merges.append(best)
        new_vocab={}
        for word,freq in vocab.items():
          new_word=[]
          i=0
          while i < len(word):
            if i < len(word)-1 and (word[i],word[i+1])==best:
              new_word.append(word[i]+word[i+1])
              i+=2
            else:
              new_word.append(word[i])
              i+=1
          new_vocab[tuple(new_word)]=freq
        vocab=new_vocab

    def encode(self, text):
        all_tokens = []
        for word in text.split():
            symbols = list(word) + ['']
            for a, b in self.merges:
                i = 0
                while i < len(symbols) - 1:
                    if symbols[i] == a and symbols[i + 1] == b:
                        symbols = symbols[:i] + [a + b] + symbols[i + 2:]
                    else:
                        i += 1
            all_tokens.extend(symbols)
        return all_tokens

In [34]:
# 🧪 Debug
bpe = SimpleBPE()
bpe.train(['low', 'low', 'low', 'lower', 'newest', 'widest'], num_merges=10)
print('Merges:', bpe.merges[:5])
print('Encode:', bpe.encode('low lower'))

Merges: [('l', 'o'), ('lo', 'w'), ('low', ''), ('e', 's'), ('es', 't')]
Encode: ['low', 'lower']


In [35]:
# ✅ SUBMIT
from torch_judge import check
check('bpe')


🧪 Testing: Byte-Pair Encoding (BPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Correct number of merges (0.2ms)
  ✅ [2/4] Most frequent pair merged first (0.1ms)
  ✅ [3/4] Encode returns list of strings (0.3ms)
  ✅ [4/4] More merges -> fewer tokens (0.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (0.8ms total)
  Progress saved. Run status() to see your dashboard.

